# M2 协议评测（v12）：运行前必须在 Kaggle UI 用 Add Input 挂载 v11 训练版本的输出

In [ ]:
# ===== v12 M2 协议评测：挂载上一版本输出 checkpoint，lerobot-eval 逐步重规划 × 10 episodes =====
# 设计原则：训练是前台进程，任何崩溃的 traceback 直接出现在批处理日志里，无静默失败。
# 无隧道/sshd/watchdog——监控走 ntfy 信标（topic smolvla-ep-b7Vq2mN4tX）。
import os, re, glob, shutil, time, subprocess, urllib.request

W = "/kaggle/working"
TOPIC = "smolvla-ep-b7Vq2mN4tX"
CODE_SHA = "146d8474932c8b566717eb2738ba459337ab71c4"
REV = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
HUB_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"
os.environ["HF_HOME"] = W + "/hf_cache"
os.makedirs(W + "/logs", exist_ok=True)

def run(cmd, timeout=1800):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def beacon(msg):
    try:
        urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + TOPIC, data=str(msg).encode()), timeout=8)
    except Exception as e:
        print("[beacon fail]", repr(e), flush=True)

# ---- 1) lerobot 安装（固定 commit）+ dtype 补丁 ----
def have_lerobot():
    return "0.6.2" in run("python -c 'import lerobot; print(lerobot.__version__)'")
if not have_lerobot():
    for cmd in ('uv pip install --system "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"',
                'pip install --no-input "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"'):
        print("[1] install ...", flush=True)
        print(run(cmd, timeout=1200)[-600:], flush=True)
        if have_lerobot():
            break
    if not have_lerobot():
        beacon("FAIL-install")
        raise RuntimeError("lerobot 安装失败（两种方式均失败，输出见上）")
run('sed -i \'s/torch_dtype="bfloat16"/torch_dtype="float32"/\' /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py')
ok = "float32" in run("grep torch_dtype /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py")
print("[1] lerobot 0.6.2 + dtype patch:", ok, flush=True)
if not ok:
    beacon("FAIL-dtype")
    raise RuntimeError("dtype 补丁失败")
beacon("OK-install")

# ---- 2) 定位上一会话训练产物（需在 Kaggle UI: Add Input -> Your Work -> 挂载本 notebook 上一版本输出）----
import glob as _glob
hits = _glob.glob("/kaggle/input/*/outputs/m1_base_libero/checkpoints/008000/pretrained_model") or        _glob.glob("/kaggle/input/*/*/outputs/m1_base_libero/checkpoints/008000/pretrained_model") or        _glob.glob("/kaggle/input/**/pretrained_model", recursive=True)
if not hits:
    beacon("FAIL-ckpt-not-found")
    raise RuntimeError("没找到 checkpoint pretrained_model，确认已在 Add Input 挂载上一版本输出")
CKPT = hits[0]
print("[2] checkpoint:", CKPT, flush=True)
beacon("OK-ckpt")

# ---- 3) 训练产物自检（权重 + processor 配置齐全）----
for f in ("model.safetensors", "policy_preprocessor.json", "train_config.json"):
    if not os.path.exists(CKPT + "/" + f):
        beacon("FAIL-ckpt-incomplete")
        raise RuntimeError("checkpoint 缺文件: " + f)
print("[3] checkpoint 自检通过", flush=True)

# ---- 4) assets + libero 配置（失败仅影响训练内评测，不致命）----
ASSETS = """import time
from pathlib import Path
from huggingface_hub import snapshot_download
import importlib.util
home = Path.home()
t0 = time.time()
snapshot_download(repo_id='lerobot/libero-assets', repo_type='dataset', local_dir='/kaggle/working/libero_assets')
spec = importlib.util.find_spec('libero')
lib_dir = Path(spec.origin).parent / 'libero'
(home / '.libero').mkdir(exist_ok=True)
lines = [
    'assets: /kaggle/working/libero_assets',
    'bddl_files: %s/bddl_files' % lib_dir,
    'datasets: %s/../datasets' % lib_dir.parent,
    'init_states: %s/init_files' % lib_dir,
]
(home / '.libero' / 'config.yaml').write_text(chr(10).join(lines) + chr(10))
print('assets %.0fs' % (time.time() - t0))
"""
open("/tmp/assets.py", "w").write(ASSETS)
out = run("python /tmp/assets.py", timeout=1800)
print("[3] assets:", out[-150:], flush=True)
if "assets" not in out:
    print("[3][warn] 资产失败，训练内评测会失败", flush=True)

# ---- 5) 数据集缓存修复（pinned revision，3 次重试）----
open("/tmp/dsfix.py", "w").write(
    "from huggingface_hub import snapshot_download\n"
    "snapshot_download('lerobot/libero', repo_type='dataset', revision='" + REV + "', max_workers=4)\n"
    "print('DS_OK')\n")
for attempt in range(3):
    out = run("python /tmp/dsfix.py 2>&1 | tail -2", timeout=2400)
    if "DS_OK" in out:
        break
    print("[5] 数据集修复重试", attempt + 1, ":", out[-200:], flush=True)
    time.sleep(20)
else:
    beacon("FAIL-dataset")
    raise RuntimeError("数据集缓存修复失败: " + out[-300:])
print("[5] 数据集缓存完整（revision " + REV[:8] + "）", flush=True)
beacon("OK-setup")

# ---- 6) 磁盘保护：只保留最新 2 个 checkpoint ----
open(W + "/prune.sh", "w").write(
    '#!/bin/bash\nD=' + W + '/outputs/m1_base_libero/checkpoints\n'
    'while true; do sleep 300; [ -d "$D" ] || continue; '
    'ls "$D" | grep -E "^[0-9]+$" | sort -n | head -n -2 | while read c; do rm -rf "$D/$c"; done; done\n')
subprocess.Popen("bash " + W + "/prune.sh", shell=True, start_new_session=True)

# ---- 5) 心跳：每 10 分钟推送进度/GPU/磁盘 ----
HB = ('#!/bin/bash\nexport PATH=/opt/bin:$PATH\nexport LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH\n'
      'while true; do sleep 600\n'
      'S=$(tail -c 400 ' + W + '/logs/eval.log | tr "\\r" "\\n" | grep -iE "episode|success" | tail -1)\n'
      'G=$(nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader 2>/dev/null | head -1)\n'
      'D=$(df -BG /kaggle/working | tail -1 | awk "{print \\$4}")\n'
      'curl -s -m 8 -d "HB $S | gpu=$G | disk_free=$D" https://ntfy.sh/' + TOPIC + ' >/dev/null\ndone\n')
open(W + "/hb.sh", "w").write(HB)
subprocess.Popen("bash " + W + "/hb.sh", shell=True, start_new_session=True)

# ---- 6) 前台评测：协议档（逐步重规划 + 官方复现参数）----
ENV = ("export PATH=/opt/bin:$PATH; export LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH; "
       "export MUJOCO_GL=egl; export HF_HOME=" + W + "/hf_cache;")
CMD = ("lerobot-eval --policy.path=" + CKPT + " "
       "--env.type=libero --env.task=libero_spatial "
       "--env.observation_height=256 --env.observation_width=256 "
       "--env.init_states=true --env.hard_reset=true "
       "--policy.n_action_steps=1 "
       "--eval.n_episodes=10 --eval.batch_size=1")
os.makedirs(W + "/logs", exist_ok=True)
MAX_TRY = 2
for attempt in range(1, MAX_TRY + 1):
    print("[6] 评测尝试 #%d (n_action_steps=1, 10 episodes)" % attempt, flush=True)
    ret = subprocess.call("bash -c 'set -o pipefail; " + ENV + " " + CMD + " 2>&1 | tee -a " + W + "/logs/eval.log'", shell=True)
    tail = run("tail -c 1500 " + W + "/logs/eval.log | tr '\\r' '\\n'")
    if ret == 0:
        import re as _re
        m = _re.search(r"(?:Success rate|success rate)[:\s]*([0-9.]+)", tail)
        rate = m.group(1) if m else tail[-200:]
        beacon("EVAL-DONE rate=" + str(rate))
        print("[6] 评测完成，成功率:", rate, flush=True)
        break
    beacon("EVAL-FAIL try%d ret=%d" % (attempt, ret))
    print("[6] 评测异常 ret=%d" % ret, flush=True)
    time.sleep(20)
else:
    beacon("EVAL-GIVEUP")
    raise RuntimeError("评测 %d 次失败，详见上方日志" % MAX_TRY)
print("ALL DONE", flush=True)
